In [ ]:

print("======================================")
print("       SOLARGUARD PROJECT")
print(" Solar Plant Underperformance Analysis")
print("======================================")

print("Google Colab is working successfully!")
print("Our project development starts now.")


       SOLARGUARD PROJECT
 Solar Plant Underperformance Analysis
Google Colab is working successfully!
Our project development starts now.


In [ ]:


# INSTALLING PROJECT LIBRARIES
#

!pip -q install streamlit pyspark plotly reportlab

print("All required SolarGuard libraries installed successfully!")


All required SolarGuard libraries installed successfully!


In [ ]:

#
# VERIFY PROJECT LIBRARIES
#

import pandas as pd
import numpy as np
import plotly
import streamlit as st
import pyspark
import reportlab

print("SOLARGUARD LIBRARY VERIFICATION")
print("--------------------------------")

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Plotly version:", plotly.__version__)
print("Streamlit version:", st.__version__)
print("PySpark version:", pyspark.__version__)

print("--------------------------------")
print("All libraries imported successfully!")

SOLARGUARD LIBRARY VERIFICATION
--------------------------------
Pandas version: 2.2.3
NumPy version: 2.4.6
Plotly version: 5.24.1
Streamlit version: 1.67.0
PySpark version: 4.0.5
--------------------------------
All libraries imported successfully!


In [ ]:


# GENERATING FIVE SIMULATED SOLAR PLANT DATASETS
#

import pandas as pd
import numpy as np
import os

# Reproducible simulated data
np.random.seed(42)

# Create project data folder
os.makedirs("SolarGuard/data", exist_ok=True)

#
# DATASET 1: INVERTER OUTPUT
#

timestamps = pd.date_range(
    start="2026-05-01 00:00:00",
    end="2026-05-07 23:45:00",
    freq="15min"
)

inverters = ["INV-01", "INV-02", "INV-03"]

inverter_records = []

for inverter in inverters:
    for timestamp in timestamps:

        hour = timestamp.hour + timestamp.minute / 60

        # Simulated sunlight pattern
        sunlight_factor = max(
            0,
            np.sin(np.pi * (hour - 6) / 12)
        )

        irradiance = 950 * sunlight_factor

        # Expected inverter output
        expected_power = 50 * (irradiance / 1000) * 0.80

        # Small natural variation
        actual_power = expected_power * np.random.uniform(0.96, 1.04)

        # Simulate underperformance in INV-02
        if (
            inverter == "INV-02"
            and timestamp.day in [4, 5]
            and 10 <= hour <= 15
        ):
            actual_power *= 0.64

        # Add small sensor variation
        actual_power += np.random.normal(0, 0.3)

        actual_power = max(0, actual_power)

        energy = actual_power * 0.25

        inverter_records.append({
            "timestamp": timestamp,
            "inverter_id": inverter,
            "ac_power_kw": round(actual_power, 3),
            "energy_kwh": round(energy, 3)
        })

inverter_df = pd.DataFrame(inverter_records)

# Introduce missing readings
inverter_df = inverter_df.drop(
    inverter_df.sample(8, random_state=42).index
)

# Introduce duplicate readings
duplicates = inverter_df.sample(5, random_state=10)
inverter_df = pd.concat(
    [inverter_df, duplicates],
    ignore_index=True
)

# Introduce one invalid negative reading for cleaning practice
inverter_df.loc[
    inverter_df.index[10], "ac_power_kw"
] = -5

# Introduce inconsistent inverter ID
inverter_df.loc[
    inverter_df.index[20], "inverter_id"
] = " inv-01 "

inverter_df.to_csv(
    "SolarGuard/data/inverter_output.csv",
    index=False
)

#
# DATASET 2: WEATHER AND IRRADIANCE
#

weather_timestamps = pd.date_range(
    start="2026-05-01 00:00:00",
    end="2026-05-07 23:30:00",
    freq="30min"
)

weather_records = []

for timestamp in weather_timestamps:

    hour = timestamp.hour + timestamp.minute / 60

    sunlight_factor = max(
        0,
        np.sin(np.pi * (hour - 6) / 12)
    )

    irradiance = 950 * sunlight_factor
    irradiance += np.random.normal(0, 15)

    irradiance = max(0, irradiance)

    ambient_temp = 25 + 8 * sunlight_factor + np.random.normal(0, 1)
    module_temp = ambient_temp + 15 * sunlight_factor

    weather_records.append({
        "timestamp": timestamp,
        "irradiance_wm2": round(irradiance, 2),
        "ambient_temperature_c": round(ambient_temp, 2),
        "module_temperature_c": round(module_temp, 2),
        "weather_condition": (
            "Sunny" if irradiance > 500
            else "Cloudy" if irradiance > 100
            else "Night/Low Light"
        )
    })

weather_df = pd.DataFrame(weather_records)

# Introduce missing weather records
weather_df = weather_df.drop(
    weather_df.sample(4, random_state=20).index
)

weather_df.to_csv(
    "SolarGuard/data/weather_irradiance.csv",
    index=False
)

#
# DATASET 3: EQUIPMENT ALARMS
#

alarms_df = pd.DataFrame([
    ["AL-001", "2026-05-04 10:30", "INV-02",
     "TEMP_HIGH", "High temperature warning", "Medium", "Open"],

    ["AL-002", "2026-05-04 12:15", "INV-02",
     "OUTPUT_LOW", "Low output warning", "High", "Open"],

    ["AL-003", "2026-05-04 14:00", "INV-02",
     "TEMP_HIGH", "High temperature warning", "Medium", "Closed"],

    ["AL-004", "2026-05-05 10:45", "INV-02",
     "OUTPUT_LOW", "Low output warning", "High", "Open"],

    ["AL-005", "2026-05-05 13:30", "INV-02",
     "TEMP_HIGH", "High temperature warning", "Medium", "Closed"],

    ["AL-006", "2026-05-02 11:00", "INV-01",
     "GRID_EVENT", "Grid fluctuation", "Low", "Closed"],

    ["AL-007", "2026-05-06 12:00", "INV-03",
     "COMM_WARNING", "Communication warning", "Low", "Closed"],

    ["AL-008", "2026-05-03 14:00", "INV-01",
     "TEMP_HIGH", "High temperature warning", "Medium", "Closed"],

    ["AL-009", "2026-05-05 15:00", "INV-02",
     "OUTPUT_LOW", "Low output warning", "High", "Closed"]
], columns=[
    "alarm_id", "timestamp", "inverter_id",
    "alarm_code", "alarm_description", "severity", "status"
])

# Introduce inconsistent description for cleaning practice
alarms_df.loc[8, "alarm_description"] = " low OUTPUT warning "

alarms_df.to_csv(
    "SolarGuard/data/equipment_alarms.csv",
    index=False
)

#
# DATASET 4: CLEANING RECORDS
#

cleaning_df = pd.DataFrame([
    ["CL-001", "2026-05-02 07:00", "INV-01", "Completed",
     "Routine panel cleaning"],

    ["CL-002", "2026-05-03 07:30", "INV-02", "Completed",
     "Routine panel cleaning"],

    ["CL-003", "2026-05-06 08:00", "INV-03", "Completed",
     "Routine panel cleaning"],

    ["CL-004", "2026-05-07 07:15", "INV-02", "Pending",
     "Scheduled cleaning"]
], columns=[
    "cleaning_id", "cleaning_timestamp", "inverter_id",
    "cleaning_status", "remarks"
])

cleaning_df.to_csv(
    "SolarGuard/data/cleaning_records.csv",
    index=False
)

#
# DATASET 5: MAINTENANCE TICKETS
#

maintenance_df = pd.DataFrame([
    ["MT-001", "INV-02", "2026-05-04 11:00",
     "2026-05-04 16:00", "Output below expected level",
     "Closed", "Inspection completed"],

    ["MT-002", "INV-02", "2026-05-05 10:00",
     None, "Repeated low output observation",
     "Open", "Technician inspection pending"],

    ["MT-003", "INV-01", "2026-05-02 13:00",
     "2026-05-02 15:00", "Grid fluctuation review",
     "Closed", "Grid connection inspected"],

    ["MT-004", "INV-03", "2026-05-06 12:30",
     "2026-05-06 14:00", "Communication warning",
     "Closed", "Communication connection checked"]
], columns=[
    "ticket_id", "inverter_id", "opened_at", "closed_at",
    "issue_description", "maintenance_status", "action_taken"
])

maintenance_df.to_csv(
    "SolarGuard/data/maintenance_tickets.csv",
    index=False
)

#
# SAVE AND DISPLAY DATASET SUMMARY
#

print("==============================================")
print("SOLARGUARD - DATASET GENERATION COMPLETED")
print("==============================================")

datasets = {
    "Inverter Output": inverter_df,
    "Weather & Irradiance": weather_df,
    "Equipment Alarms": alarms_df,
    "Cleaning Records": cleaning_df,
    "Maintenance Tickets": maintenance_df
}

for name, df in datasets.items():
    print(f"{name}: {len(df)} records")

print("\nAll five CSV files saved in SolarGuard/data/")
print("==============================================")


SOLARGUARD - DATASET GENERATION COMPLETED
Inverter Output: 2013 records
Weather & Irradiance: 332 records
Equipment Alarms: 9 records
Cleaning Records: 4 records
Maintenance Tickets: 4 records

All five CSV files saved in SolarGuard/data/


In [ ]:

#
# STEP 3.3 - PREVIEW ALL FIVE DATASETS
#

print("1. INVERTER OUTPUT")
display(inverter_df.head())

print("2. WEATHER AND IRRADIANCE")
display(weather_df.head())

print("3. EQUIPMENT ALARMS")
display(alarms_df.head())

print("4. CLEANING RECORDS")
display(cleaning_df.head())

print("5. MAINTENANCE TICKETS")
display(maintenance_df.head())


1. INVERTER OUTPUT


,timestamp,inverter_id,ac_power_kw,energy_kwh
0,2026-05-01 00:00:00,INV-01,0.000,0.000
1,2026-05-01 00:15:00,INV-01,0.096,0.024
2,2026-05-01 00:30:00,INV-01,0.084,0.021
3,2026-05-01 00:45:00,INV-01,0.303,0.076
4,2026-05-01 01:00:00,INV-01,0.000,0.000


2. WEATHER AND IRRADIANCE


,timestamp,irradiance_wm2,ambient_temperature_c,module_temperature_c,weather_condition
0,2026-05-01 00:00:00,0.00,24.72,24.72,Night/Low Light
1,2026-05-01 00:30:00,17.92,25.89,25.89,Night/Low Light
2,2026-05-01 01:00:00,0.00,25.35,25.35,Night/Low Light
3,2026-05-01 01:30:00,0.00,25.05,25.05,Night/Low Light
4,2026-05-01 02:00:00,28.47,24.05,24.05,Night/Low Light


3. EQUIPMENT ALARMS


,alarm_id,timestamp,inverter_id,alarm_code,alarm_description,severity,status
0,AL-001,2026-05-04 10:30,INV-02,TEMP_HIGH,High temperature warning,Medium,Open
1,AL-002,2026-05-04 12:15,INV-02,OUTPUT_LOW,Low output warning,High,Open
2,AL-003,2026-05-04 14:00,INV-02,TEMP_HIGH,High temperature warning,Medium,Closed
3,AL-004,2026-05-05 10:45,INV-02,OUTPUT_LOW,Low output warning,High,Open
4,AL-005,2026-05-05 13:30,INV-02,TEMP_HIGH,High temperature warning,Medium,Closed


4. CLEANING RECORDS


,cleaning_id,cleaning_timestamp,inverter_id,cleaning_status,remarks
0,CL-001,2026-05-02 07:00,INV-01,Completed,Routine panel cleaning
1,CL-002,2026-05-03 07:30,INV-02,Completed,Routine panel cleaning
2,CL-003,2026-05-06 08:00,INV-03,Completed,Routine panel cleaning
3,CL-004,2026-05-07 07:15,INV-02,Pending,Scheduled cleaning


5. MAINTENANCE TICKETS


,ticket_id,inverter_id,opened_at,closed_at,issue_description,maintenance_status,action_taken
0,MT-001,INV-02,2026-05-04 11:00,2026-05-04 16:00,Output below expected level,Closed,Inspection completed
1,MT-002,INV-02,2026-05-05 10:00,None,Repeated low output observation,Open,Technician inspection pending
2,MT-003,INV-01,2026-05-02 13:00,2026-05-02 15:00,Grid fluctuation review,Closed,Grid connection inspected
3,MT-004,INV-03,2026-05-06 12:30,2026-05-06 14:00,Communication warning,Closed,Communication connection checked


In [ ]:



# CORRECT NIGHTTIME SENSOR READINGS


# Convert timestamps to datetime
inverter_df["timestamp"] = pd.to_datetime(
    inverter_df["timestamp"]
)

weather_df["timestamp"] = pd.to_datetime(
    weather_df["timestamp"]
)

# Identify nighttime readings
night_weather = (
    (weather_df["timestamp"].dt.hour < 6) |
    (weather_df["timestamp"].dt.hour >= 18)
)

night_inverter = (
    (inverter_df["timestamp"].dt.hour < 6) |
    (inverter_df["timestamp"].dt.hour >= 18)
)

# Set nighttime irradiance to zero
weather_df.loc[night_weather, "irradiance_wm2"] = 0

# Set nighttime inverter power and energy to zero
inverter_df.loc[night_inverter, "ac_power_kw"] = 0
inverter_df.loc[night_inverter, "energy_kwh"] = 0

# Preserve one intentional invalid daytime reading
invalid_row = inverter_df[
    (inverter_df["inverter_id"] == "INV-01") &
    (inverter_df["timestamp"] == "2026-05-01 12:00:00")
].index

if len(invalid_row) > 0:
    inverter_df.loc[invalid_row[0], "ac_power_kw"] = -5
    inverter_df.loc[invalid_row[0], "energy_kwh"] = -1.25

# Save corrected datasets
inverter_df.to_csv(
    "SolarGuard/data/inverter_output.csv",
    index=False
)

weather_df.to_csv(
    "SolarGuard/data/weather_irradiance.csv",
    index=False
)

print("Nighttime readings corrected successfully!")
print("One invalid daytime reading preserved for cleaning practice.")

print("\nNighttime irradiance:")
display(weather_df.loc[night_weather].head())

print("\nInvalid daytime inverter reading:")
display(inverter_df.loc[invalid_row])


Nighttime readings corrected successfully!
One invalid daytime reading preserved for cleaning practice.

Nighttime irradiance:


,timestamp,irradiance_wm2,ambient_temperature_c,module_temperature_c,weather_condition
0,2026-05-01 00:00:00,0.0,24.72,24.72,Night/Low Light
1,2026-05-01 00:30:00,0.0,25.89,25.89,Night/Low Light
2,2026-05-01 01:00:00,0.0,25.35,25.35,Night/Low Light
3,2026-05-01 01:30:00,0.0,25.05,25.05,Night/Low Light
4,2026-05-01 02:00:00,0.0,24.05,24.05,Night/Low Light



Invalid daytime inverter reading:


,timestamp,inverter_id,ac_power_kw,energy_kwh
48,2026-05-01 12:00:00,INV-01,-5.0,-1.25


In [ ]:



# DATA EXPLORATION AND DATA QUALITY AUDIT


print("SOLARGUARD - DATA QUALITY AUDIT")
print("=" * 55)

all_datasets = {
    "Inverter Output": inverter_df,
    "Weather & Irradiance": weather_df,
    "Equipment Alarms": alarms_df,
    "Cleaning Records": cleaning_df,
    "Maintenance Tickets": maintenance_df
}

# 1. Basic information about every dataset
for name, df in all_datasets.items():

    print("\nDATASET:", name)
    print("-" * 45)

    print("Total records:", len(df))
    print("Total columns:", len(df.columns))

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)

# 2. Find invalid inverter readings
print("\n" + "=" * 55)
print("INVALID INVERTER POWER READINGS")

invalid_power = inverter_df[
    inverter_df["ac_power_kw"] < 0
]

display(invalid_power)

# 3. Find inconsistent inverter identifiers
print("\nUNIQUE INVERTER IDENTIFIERS")

print(inverter_df["inverter_id"].unique())

# 4. Find duplicate inverter observations
print("\nDUPLICATE INVERTER OBSERVATIONS")

duplicate_readings = inverter_df[
    inverter_df.duplicated(
        subset=["timestamp", "inverter_id"],
        keep=False
    )
]

display(duplicate_readings)

# 5. Summary statistics
print("\nINVERTER POWER STATISTICS")

display(inverter_df["ac_power_kw"].describe())

print("\n" + "=" * 55)
print("DATA QUALITY AUDIT COMPLETED")


SOLARGUARD - DATA QUALITY AUDIT

DATASET: Inverter Output
---------------------------------------------
Total records: 2013
Total columns: 4

Missing values:
timestamp      0
inverter_id    0
ac_power_kw    0
energy_kwh     0
dtype: int64

Duplicate rows: 5

Data types:
timestamp      datetime64[ns]
inverter_id            object
ac_power_kw           float64
energy_kwh            float64
dtype: object

DATASET: Weather & Irradiance
---------------------------------------------
Total records: 332
Total columns: 5

Missing values:
timestamp                0
irradiance_wm2           0
ambient_temperature_c    0
module_temperature_c     0
weather_condition        0
dtype: int64

Duplicate rows: 0

Data types:
timestamp                datetime64[ns]
irradiance_wm2                  float64
ambient_temperature_c           float64
module_temperature_c            float64
weather_condition                object
dtype: object

DATASET: Equipment Alarms
--------------------------------------------

,timestamp,inverter_id,ac_power_kw,energy_kwh
48,2026-05-01 12:00:00,INV-01,-5.0,-1.25



UNIQUE INVERTER IDENTIFIERS
['INV-01' ' inv-01 ' 'INV-02' 'INV-03']

DUPLICATE INVERTER OBSERVATIONS


,timestamp,inverter_id,ac_power_kw,energy_kwh
36,2026-05-01 09:00:00,INV-01,27.391,6.848
163,2026-05-02 16:45:00,INV-01,12.686,3.171
212,2026-05-03 05:00:00,INV-01,0.000,0.000
879,2026-05-03 04:45:00,INV-02,0.000,0.000
898,2026-05-03 09:30:00,INV-02,31.317,7.829
2008,2026-05-03 04:45:00,INV-02,0.000,0.000
2009,2026-05-01 09:00:00,INV-01,27.391,6.848
2010,2026-05-02 16:45:00,INV-01,12.686,3.171
2011,2026-05-03 09:30:00,INV-02,31.317,7.829
2012,2026-05-03 05:00:00,INV-01,0.000,0.000



INVERTER POWER STATISTICS


,ac_power_kw
count,2013.000000
mean,11.771546
std,14.339422
min,-5.000000
25%,0.000000
50%,0.000000
75%,24.936000
max,39.731000



DATA QUALITY AUDIT COMPLETED


In [ ]:


# DATA CLEANING USING PANDAS
#

print("SOLARGUARD DATA CLEANING STARTED")
print("=" * 50)

# Make working copies of raw datasets
clean_inverter = inverter_df.copy()
clean_weather = weather_df.copy()
clean_alarms = alarms_df.copy()
clean_cleaning = cleaning_df.copy()
clean_maintenance = maintenance_df.copy()

#
# 1. STANDARDIZE INVERTER IDENTIFIERS
#

clean_inverter["inverter_id"] = (
    clean_inverter["inverter_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

#
# 2. REMOVE DUPLICATE INVERTER OBSERVATIONS
#

before_duplicates = len(clean_inverter)

clean_inverter = clean_inverter.drop_duplicates(
    subset=["timestamp", "inverter_id"],
    keep="first"
)

removed_duplicates = before_duplicates - len(clean_inverter)

#
# 3. HANDLE INVALID NEGATIVE POWER READINGS
#

invalid_count = (
    clean_inverter["ac_power_kw"] < 0
).sum()

# Replace invalid power and energy with missing values
clean_inverter.loc[
    clean_inverter["ac_power_kw"] < 0,
    ["ac_power_kw", "energy_kwh"]
] = np.nan

# Preserve a flag for invalid measurements
clean_inverter["invalid_reading_flag"] = (
    clean_inverter["ac_power_kw"].isna()
)

#
# 4. STANDARDIZE WEATHER DATA
#

clean_weather["weather_condition"] = (
    clean_weather["weather_condition"]
    .astype(str)
    .str.strip()
)

#
# 5. STANDARDIZE ALARM DESCRIPTIONS
#

clean_alarms["alarm_description"] = (
    clean_alarms["alarm_description"]
    .astype(str)
    .str.strip()
    .str.title()
)

clean_alarms["inverter_id"] = (
    clean_alarms["inverter_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

#
# 6. HANDLE MISSING MAINTENANCE CLOSING TIME
#

clean_maintenance["opened_at"] = pd.to_datetime(
    clean_maintenance["opened_at"]
)

clean_maintenance["closed_at"] = pd.to_datetime(
    clean_maintenance["closed_at"]
)

# Keep open ticket closing times missing.
# Do not invent a closing date.

#
# 7. DISPLAY CLEANING RESULTS
#

print("\nDATA CLEANING SUMMARY")
print("-" * 40)

print("Duplicate rows removed:", removed_duplicates)
print("Invalid power readings identified:", invalid_count)

print("Remaining inverter records:", len(clean_inverter))

print(
    "Missing inverter power values:",
    clean_inverter["ac_power_kw"].isna().sum()
)

print(
    "Unique inverter identifiers:",
    clean_inverter["inverter_id"].unique()
)

#
# 8. SAVE CLEANED DATASETS
#

clean_inverter.to_csv(
    "SolarGuard/data/clean_inverter_output.csv",
    index=False
)

clean_weather.to_csv(
    "SolarGuard/data/clean_weather_irradiance.csv",
    index=False
)

clean_alarms.to_csv(
    "SolarGuard/data/clean_equipment_alarms.csv",
    index=False
)

clean_cleaning.to_csv(
    "SolarGuard/data/clean_cleaning_records.csv",
    index=False
)

clean_maintenance.to_csv(
    "SolarGuard/data/clean_maintenance_tickets.csv",
    index=False
)

print("\nAll cleaned datasets saved successfully!")
print("STEP 5 COMPLETED")


SOLARGUARD DATA CLEANING STARTED

DATA CLEANING SUMMARY
----------------------------------------
Duplicate rows removed: 5
Invalid power readings identified: 1
Remaining inverter records: 2008
Missing inverter power values: 1
Unique inverter identifiers: ['INV-01' 'INV-02' 'INV-03']

All cleaned datasets saved successfully!
STEP 5 COMPLETED


In [ ]:

#
# VERIFY CLEANED INVERTER DATA
#

print("CLEANED INVERTER DATA")
display(clean_inverter.head(10))

print("\nCHECK DUPLICATES")

print(
    clean_inverter.duplicated(
        subset=["timestamp", "inverter_id"]
    ).sum()
)

print("\nCHECK INVALID READINGS")

display(
    clean_inverter[
        clean_inverter["invalid_reading_flag"]
    ]
)

print("\nCHECK INVERTER IDENTIFIERS")

print(clean_inverter["inverter_id"].unique())



CLEANED INVERTER DATA


,timestamp,inverter_id,ac_power_kw,energy_kwh,invalid_reading_flag
0,2026-05-01 00:00:00,INV-01,0.0,0.0,False
1,2026-05-01 00:15:00,INV-01,0.0,0.0,False
2,2026-05-01 00:30:00,INV-01,0.0,0.0,False
3,2026-05-01 00:45:00,INV-01,0.0,0.0,False
4,2026-05-01 01:00:00,INV-01,0.0,0.0,False
5,2026-05-01 01:15:00,INV-01,0.0,0.0,False
6,2026-05-01 01:30:00,INV-01,0.0,0.0,False
7,2026-05-01 01:45:00,INV-01,0.0,0.0,False
8,2026-05-01 02:00:00,INV-01,0.0,0.0,False
9,2026-05-01 02:15:00,INV-01,0.0,0.0,False



CHECK DUPLICATES
0

CHECK INVALID READINGS


,timestamp,inverter_id,ac_power_kw,energy_kwh,invalid_reading_flag
48,2026-05-01 12:00:00,INV-01,NaN,NaN,True



CHECK INVERTER IDENTIFIERS
['INV-01' 'INV-02' 'INV-03']


In [ ]:


# TIME SERIES ALIGNMENT AND MISSING READING DETECTION
#

print("SOLARGUARD TIME SERIES ALIGNMENT")
print("=" * 55)

# Ensure timestamps have correct datetime format
clean_inverter["timestamp"] = pd.to_datetime(
    clean_inverter["timestamp"]
)

clean_weather["timestamp"] = pd.to_datetime(
    clean_weather["timestamp"]
)

#
# 1. CREATE A COMPLETE 15-MINUTE TIMELINE
#

full_timeline = pd.date_range(
    start="2026-05-01 00:00:00",
    end="2026-05-07 23:45:00",
    freq="15min"
)

inverter_list = ["INV-01", "INV-02", "INV-03"]

# Generate every expected timestamp for each inverter
complete_grid = pd.MultiIndex.from_product(
    [inverter_list, full_timeline],
    names=["inverter_id", "timestamp"]
).to_frame(index=False)

print("Expected inverter observations:", len(complete_grid))

#
# 2. ALIGN INVERTER DATA WITH COMPLETE TIMELINE
#

aligned_df = complete_grid.merge(
    clean_inverter,
    on=["inverter_id", "timestamp"],
    how="left",
    indicator=True
)

# Identify missing inverter records
aligned_df["missing_inverter_reading"] = (
    aligned_df["_merge"] == "left_only"
)

aligned_df = aligned_df.drop(columns=["_merge"])

# Preserve invalid measurement flags
aligned_df["invalid_reading_flag"] = (
    aligned_df["invalid_reading_flag"]
    .fillna(False)
)

#
# 3. ALIGN WEATHER DATA TO 15-MINUTE TIMELINE
#

# Record whether an exact weather observation exists
weather_times = set(clean_weather["timestamp"])

aligned_df["exact_weather_match"] = (
    aligned_df["timestamp"].isin(weather_times)
)

# Sort data before merge_asof
aligned_df = aligned_df.sort_values("timestamp")

weather_sorted = clean_weather.sort_values("timestamp")

# Use nearest weather observation within 15 minutes
aligned_df = pd.merge_asof(
    aligned_df,
    weather_sorted,
    on="timestamp",
    direction="nearest",
    tolerance=pd.Timedelta("15min")
)

# Identify missing weather context
aligned_df["missing_weather_observation"] = (
    ~aligned_df["exact_weather_match"]
)

#
# 4. SORT FINAL ALIGNED DATASET
#

aligned_df = aligned_df.sort_values(
    ["timestamp", "inverter_id"]
).reset_index(drop=True)

#
# 5. DISPLAY SUMMARY
#

print("\nTIME SERIES ALIGNMENT SUMMARY")
print("-" * 45)

print("Total aligned records:", len(aligned_df))

print(
    "Missing inverter readings:",
    aligned_df["missing_inverter_reading"].sum()
)

print(
    "Missing exact weather observations:",
    aligned_df["missing_weather_observation"].sum()
)

print(
    "Invalid inverter measurements:",
    aligned_df["invalid_reading_flag"].sum()
)

#
# 6. SAVE ALIGNED DATASET
#

aligned_df.to_csv(
    "SolarGuard/data/solar_aligned_data.csv",
    index=False
)

print("\nAligned dataset saved successfully!")
print("File: SolarGuard/data/solar_aligned_data.csv")

print("\nALIGNED DATA PREVIEW")
display(aligned_df.head(10))

print("\nMISSING INVERTER READING DETAILS")
display(
    aligned_df[
        aligned_df["missing_inverter_reading"]
    ].head(10)
)

print("\nSTEP 6 COMPLETED")


SOLARGUARD TIME SERIES ALIGNMENT
Expected inverter observations: 2016

TIME SERIES ALIGNMENT SUMMARY
---------------------------------------------
Total aligned records: 2016
Missing inverter readings: 8
Missing exact weather observations: 1020
Invalid inverter measurements: 1

Aligned dataset saved successfully!
File: SolarGuard/data/solar_aligned_data.csv

ALIGNED DATA PREVIEW


/tmp/ipykernel_681/1835995427.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,inverter_id,timestamp,ac_power_kw,energy_kwh,invalid_reading_flag,missing_inverter_reading,exact_weather_match,irradiance_wm2,ambient_temperature_c,module_temperature_c,weather_condition,missing_weather_observation
0,INV-01,2026-05-01 00:00:00,0.0,0.0,False,False,True,0.0,24.72,24.72,Night/Low Light,False
1,INV-02,2026-05-01 00:00:00,0.0,0.0,False,False,True,0.0,24.72,24.72,Night/Low Light,False
2,INV-03,2026-05-01 00:00:00,0.0,0.0,False,False,True,0.0,24.72,24.72,Night/Low Light,False
3,INV-01,2026-05-01 00:15:00,0.0,0.0,False,False,False,0.0,24.72,24.72,Night/Low Light,True
4,INV-02,2026-05-01 00:15:00,0.0,0.0,False,False,False,0.0,24.72,24.72,Night/Low Light,True
5,INV-03,2026-05-01 00:15:00,0.0,0.0,False,False,False,0.0,24.72,24.72,Night/Low Light,True
6,INV-01,2026-05-01 00:30:00,0.0,0.0,False,False,True,0.0,25.89,25.89,Night/Low Light,False
7,INV-02,2026-05-01 00:30:00,0.0,0.0,False,False,True,0.0,25.89,25.89,Night/Low Light,False
8,INV-03,2026-05-01 00:30:00,0.0,0.0,False,False,True,0.0,25.89,25.89,Night/Low Light,False
9,INV-01,2026-05-01 00:45:00,0.0,0.0,False,False,False,0.0,25.89,25.89,Night/Low Light,True



MISSING INVERTER READING DETAILS


,inverter_id,timestamp,ac_power_kw,energy_kwh,invalid_reading_flag,missing_inverter_reading,exact_weather_match,irradiance_wm2,ambient_temperature_c,module_temperature_c,weather_condition,missing_weather_observation
191,INV-03,2026-05-01 15:45:00,NaN,NaN,False,True,False,596.20,29.79,38.92,Sunny,True
1130,INV-03,2026-05-04 22:00:00,NaN,NaN,False,True,True,0.00,25.75,25.75,Night/Low Light,False
1179,INV-01,2026-05-05 02:15:00,NaN,NaN,False,True,False,0.00,25.63,25.63,Night/Low Light,True
1255,INV-02,2026-05-05 08:30:00,NaN,NaN,False,True,True,566.18,30.83,39.96,Sunny,False
1287,INV-01,2026-05-05 11:15:00,NaN,NaN,False,True,False,931.62,32.31,46.80,Sunny,True
1299,INV-01,2026-05-05 12:15:00,NaN,NaN,False,True,False,948.43,33.39,48.39,Sunny,True
1578,INV-01,2026-05-06 11:30:00,NaN,NaN,False,True,True,947.59,33.24,48.11,Sunny,False
1579,INV-02,2026-05-06 11:30:00,NaN,NaN,False,True,True,947.59,33.24,48.11,Sunny,False



STEP 6 COMPLETED


In [ ]:

#
#
# EXPECTED OUTPUT AND UNDERPERFORMANCE DETECTION
#

print("SOLARGUARD PERFORMANCE ANALYSIS")
print("=" * 60)

#
# 1. DEFINE BASELINE PARAMETERS
#

RATED_CAPACITY_KW = 50
PERFORMANCE_RATIO = 0.80
DEFICIT_THRESHOLD = 20

# Make a working copy
performance_df = aligned_df.copy()

#
# 2. CALCULATE EXPECTED POWER
#

performance_df["expected_power_kw"] = (
    RATED_CAPACITY_KW *
    (performance_df["irradiance_wm2"] / 1000) *
    PERFORMANCE_RATIO
)

# Expected energy for a 15-minute interval
performance_df["expected_energy_kwh"] = (
    performance_df["expected_power_kw"] * 0.25
)

#
# 3. CALCULATE ACTUAL ENERGY AND DEFICIT
#

performance_df["actual_energy_kwh"] = (
    performance_df["ac_power_kw"] * 0.25
)

performance_df["power_deficit_kw"] = (
    performance_df["expected_power_kw"] -
    performance_df["ac_power_kw"]
)

performance_df["energy_deficit_kwh"] = (
    performance_df["expected_energy_kwh"] -
    performance_df["actual_energy_kwh"]
)

# Calculate percentage deficit
performance_df["deficit_percentage"] = np.where(
    performance_df["expected_power_kw"] > 0,
    (
        performance_df["power_deficit_kw"] /
        performance_df["expected_power_kw"]
    ) * 100,
    np.nan
)

#
# 4. CLASSIFY EACH OBSERVATION
#

conditions = [
    performance_df["missing_inverter_reading"],

    performance_df["invalid_reading_flag"],

    performance_df["irradiance_wm2"].isna(),

    performance_df["irradiance_wm2"] < 100,

    (
        performance_df["ac_power_kw"].notna() &
        (performance_df["expected_power_kw"] > 0) &
        (
            performance_df["deficit_percentage"]
            >= DEFICIT_THRESHOLD
        )
    )
]

labels = [
    "Missing Telemetry",
    "Invalid Reading",
    "Missing Weather Context",
    "Low Irradiance / Not Evaluated",
    "Underperformance"
]

performance_df["performance_status"] = np.select(
    conditions,
    labels,
    default="Normal Operation"
)

#
# 5. DISPLAY STATUS COUNTS
#

print("\nPERFORMANCE STATUS SUMMARY")
print("-" * 45)

print(
    performance_df["performance_status"]
    .value_counts()
)

#
# 6. IDENTIFY UNDERPERFORMANCE RECORDS
#

underperformance_df = performance_df[
    performance_df["performance_status"] ==
    "Underperformance"
].copy()

print("\nUNDERPERFORMANCE RECORDS")
print("-" * 45)

print("Total flagged observations:", len(underperformance_df))

display(
    underperformance_df[
        [
            "timestamp",
            "inverter_id",
            "irradiance_wm2",
            "expected_power_kw",
            "ac_power_kw",
            "deficit_percentage",
            "performance_status"
        ]
    ].head(15)
)

#
# 7. CALCULATE SUMMARY MEASURES
#

valid_df = performance_df[
    performance_df["performance_status"].isin(
        ["Normal Operation", "Underperformance"]
    )
].copy()

expected_total = valid_df["expected_energy_kwh"].sum()
actual_total = valid_df["actual_energy_kwh"].sum()

total_shortfall = max(
    0,
    expected_total - actual_total
)

underperformance_hours = (
    len(underperformance_df) * 0.25
)

print("\nKEY PERFORMANCE MEASURES")
print("-" * 45)

print("Expected energy:", round(expected_total, 2), "kWh")
print("Actual energy:", round(actual_total, 2), "kWh")
print("Positive energy shortfall:", round(total_shortfall, 2), "kWh")
print("Flagged inverter-hours:", round(underperformance_hours, 2))

#
# 8. SAVE PERFORMANCE RESULTS
#

performance_df.to_csv(
    "SolarGuard/data/solar_performance_analysis.csv",
    index=False
)

underperformance_df.to_csv(
    "SolarGuard/data/underperformance_records.csv",
    index=False
)

print("\nPerformance analysis saved successfully!")
print("STEP 7 COMPLETED")


SOLARGUARD PERFORMANCE ANALYSIS

PERFORMANCE STATUS SUMMARY
---------------------------------------------
performance_status
Low Irradiance / Not Evaluated    1042
Normal Operation                   865
Underperformance                    88
Missing Weather Context             12
Missing Telemetry                    8
Invalid Reading                      1
Name: count, dtype: int64

UNDERPERFORMANCE RECORDS
---------------------------------------------
Total flagged observations: 88


,timestamp,inverter_id,irradiance_wm2,expected_power_kw,ac_power_kw,deficit_percentage,performance_status
207,2026-05-01 17:15:00,INV-01,254.92,10.1968,7.781,23.691746,Underperformance
208,2026-05-01 17:15:00,INV-02,254.92,10.1968,7.066,30.703750,Underperformance
209,2026-05-01 17:15:00,INV-03,254.92,10.1968,7.851,23.005257,Underperformance
211,2026-05-01 17:30:00,INV-02,161.28,6.4512,4.973,22.913566,Underperformance
212,2026-05-01 17:30:00,INV-03,161.28,6.4512,4.901,24.029638,Underperformance
213,2026-05-01 17:45:00,INV-01,161.28,6.4512,2.910,54.892113,Underperformance
214,2026-05-01 17:45:00,INV-02,161.28,6.4512,2.210,65.742808,Underperformance
215,2026-05-01 17:45:00,INV-03,161.28,6.4512,2.177,66.254340,Underperformance
495,2026-05-02 17:15:00,INV-01,243.62,9.7448,7.382,24.246778,Underperformance
496,2026-05-02 17:15:00,INV-02,243.62,9.7448,7.342,24.657253,Underperformance



KEY PERFORMANCE MEASURES
---------------------------------------------
Expected energy: 5999.54 kWh
Actual energy: 5860.22 kWh
Positive energy shortfall: 139.32 kWh
Flagged inverter-hours: 22.0

Performance analysis saved successfully!
STEP 7 COMPLETED


In [ ]:


# EQUIPMENT ALARM AND MAINTENANCE CORRELATION
#

print("SOLARGUARD EVENT CORRELATION ANALYSIS")
print("=" * 60)

# Create working copies
events_df = performance_df.copy()

# Convert event timestamps
clean_alarms["timestamp"] = pd.to_datetime(
    clean_alarms["timestamp"]
)

clean_cleaning["cleaning_timestamp"] = pd.to_datetime(
    clean_cleaning["cleaning_timestamp"]
)

#
# 1. CHECK ALARMS WITHIN 60 MINUTES OF EACH READING
#

events_df["alarm_within_60min"] = False
events_df["nearby_alarm_count"] = 0

for idx, row in events_df.iterrows():

    matching_alarms = clean_alarms[
        (clean_alarms["inverter_id"] == row["inverter_id"]) &
        (
            (
                clean_alarms["timestamp"] -
                row["timestamp"]
            ).abs() <= pd.Timedelta("60min")
        )
    ]

    events_df.loc[idx, "nearby_alarm_count"] = len(
        matching_alarms
    )

    events_df.loc[idx, "alarm_within_60min"] = (
        len(matching_alarms) > 0
    )

#
# 2. CHECK ACTIVE MAINTENANCE TICKETS
#

events_df["maintenance_overlap"] = False
events_df["related_ticket_ids"] = ""

monitoring_end = events_df["timestamp"].max()

for idx, row in events_df.iterrows():

    matching_tickets = clean_maintenance[
        clean_maintenance["inverter_id"] == row["inverter_id"]
    ]

    active_tickets = []

    for _, ticket in matching_tickets.iterrows():

        ticket_end = (
            ticket["closed_at"]
            if pd.notna(ticket["closed_at"])
            else monitoring_end
        )

        if (
            ticket["opened_at"] <= row["timestamp"] <= ticket_end
        ):
            active_tickets.append(ticket["ticket_id"])

    events_df.loc[idx, "maintenance_overlap"] = (
        len(active_tickets) > 0
    )

    events_df.loc[idx, "related_ticket_ids"] = ", ".join(
        active_tickets
    )

#
# 3. CHECK RECENT CLEANING RECORDS
#

events_df["cleaning_within_24h"] = False

for idx, row in events_df.iterrows():

    matching_cleaning = clean_cleaning[
        (clean_cleaning["inverter_id"] == row["inverter_id"]) &
        (clean_cleaning["cleaning_status"] == "Completed") &
        (
            (
                clean_cleaning["cleaning_timestamp"] -
                row["timestamp"]
            ).abs() <= pd.Timedelta("24h")
        )
    ]

    events_df.loc[idx, "cleaning_within_24h"] = (
        len(matching_cleaning) > 0
    )

#
# 4. SELECT UNDERPERFORMANCE EVENTS
#

investigation_df = events_df[
    events_df["performance_status"] == "Underperformance"
].copy()

investigation_df["event_correlation"] = np.select(
    [
        (
            investigation_df["alarm_within_60min"] &
            investigation_df["maintenance_overlap"]
        ),
        investigation_df["alarm_within_60min"],
        investigation_df["maintenance_overlap"]
    ],
    [
        "Alarm and Maintenance Overlap",
        "Alarm Nearby",
        "Maintenance Overlap"
    ],
    default="No Recorded Event Nearby"
)

#
# 5. DISPLAY INVESTIGATION RESULTS
#

print("\nUNDERPERFORMANCE EVENT INVESTIGATION")
print("-" * 55)

display(
    investigation_df[
        [
            "timestamp",
            "inverter_id",
            "ac_power_kw",
            "expected_power_kw",
            "deficit_percentage",
            "nearby_alarm_count",
            "alarm_within_60min",
            "maintenance_overlap",
            "cleaning_within_24h",
            "event_correlation"
        ]
    ].head(20)
)

#
# 6. CREATE INVERTER-WISE EVENT SUMMARY
#

event_summary = investigation_df.groupby(
    "inverter_id"
).agg(
    flagged_observations=(
        "timestamp", "count"
    ),
    observations_near_alarm=(
        "alarm_within_60min", "sum"
    ),
    observations_during_maintenance=(
        "maintenance_overlap", "sum"
    ),
    observations_near_cleaning=(
        "cleaning_within_24h", "sum"
    ),
    average_deficit_percentage=(
        "deficit_percentage", "mean"
    )
).reset_index()

print("\nEVENT CORRELATION SUMMARY")

display(event_summary.round(2))

# ------------------------------------------------------------
# 7. SAVE INVESTIGATION RESULTS
# ------------------------------------------------------------

investigation_df.to_csv(
    "SolarGuard/data/investigation_events.csv",
    index=False
)

event_summary.to_csv(
    "SolarGuard/data/event_correlation_summary.csv",
    index=False
)

print("\nInvestigation records saved successfully!")
print("STEP 8 COMPLETED")


SOLARGUARD EVENT CORRELATION ANALYSIS

UNDERPERFORMANCE EVENT INVESTIGATION
-------------------------------------------------------


,timestamp,inverter_id,ac_power_kw,expected_power_kw,deficit_percentage,nearby_alarm_count,alarm_within_60min,maintenance_overlap,cleaning_within_24h,event_correlation
207,2026-05-01 17:15:00,INV-01,7.781,10.1968,23.691746,0,False,False,True,No Recorded Event Nearby
208,2026-05-01 17:15:00,INV-02,7.066,10.1968,30.703750,0,False,False,False,No Recorded Event Nearby
209,2026-05-01 17:15:00,INV-03,7.851,10.1968,23.005257,0,False,False,False,No Recorded Event Nearby
211,2026-05-01 17:30:00,INV-02,4.973,6.4512,22.913566,0,False,False,False,No Recorded Event Nearby
212,2026-05-01 17:30:00,INV-03,4.901,6.4512,24.029638,0,False,False,False,No Recorded Event Nearby
213,2026-05-01 17:45:00,INV-01,2.910,6.4512,54.892113,0,False,False,True,No Recorded Event Nearby
214,2026-05-01 17:45:00,INV-02,2.210,6.4512,65.742808,0,False,False,False,No Recorded Event Nearby
215,2026-05-01 17:45:00,INV-03,2.177,6.4512,66.254340,0,False,False,False,No Recorded Event Nearby
495,2026-05-02 17:15:00,INV-01,7.382,9.7448,24.246778,0,False,False,True,No Recorded Event Nearby
496,2026-05-02 17:15:00,INV-02,7.342,9.7448,24.657253,0,False,False,True,No Recorded Event Nearby



EVENT CORRELATION SUMMARY


,inverter_id,flagged_observations,observations_near_alarm,observations_during_maintenance,observations_near_cleaning,average_deficit_percentage
0,INV-01,15,0,0,5,38.53
1,INV-02,57,40,43,5,36.71
2,INV-03,16,0,0,4,36.97



Investigation records saved successfully!
STEP 8 COMPLETED


In [ ]:
# FIX FOR STEP 9: CREATE INVERTER SUMMARY

inverter_summary = performance_df.groupby(
    "inverter_id"
).agg(
    expected_energy_kwh=("expected_energy_kwh", "sum"),
    actual_energy_kwh=("actual_energy_kwh", "sum")
).reset_index()

inverter_summary["energy_shortfall_kwh"] = (
    inverter_summary["expected_energy_kwh"] -
    inverter_summary["actual_energy_kwh"]
).clip(lower=0)

inverter_summary["energy_deficit_percentage"] = (
    inverter_summary["energy_shortfall_kwh"] /
    inverter_summary["expected_energy_kwh"].replace(0, np.nan)
) * 100

print("INVERTER SUMMARY CREATED SUCCESSFULLY!")

display(inverter_summary.round(2))

INVERTER SUMMARY CREATED SUCCESSFULLY!


,inverter_id,expected_energy_kwh,actual_energy_kwh,energy_shortfall_kwh,energy_deficit_percentage
0,INV-01,2020.4,1995.35,25.05,1.24
1,INV-02,2020.4,1886.74,133.66,6.62
2,INV-03,2020.4,2025.34,0.00,0.00


In [ ]:

#
# INTERACTIVE DATA VISUALIZATION USING PLOTLY
#

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("SOLARGUARD VISUALIZATION MODULE")
print("=" * 55)

#
# CHART 1: ACTUAL VS EXPECTED SOLAR GENERATION
#

plant_timeline = performance_df.groupby(
    "timestamp"
).agg(
    actual_power_kw=("ac_power_kw", "sum"),
    expected_power_kw=("expected_power_kw", "sum")
).reset_index()

fig1 = go.Figure()

fig1.add_trace(
    go.Scatter(
        x=plant_timeline["timestamp"],
        y=plant_timeline["expected_power_kw"],
        mode="lines",
        name="Expected Power",
        line=dict(color="royalblue", dash="dash")
    )
)

fig1.add_trace(
    go.Scatter(
        x=plant_timeline["timestamp"],
        y=plant_timeline["actual_power_kw"],
        mode="lines",
        name="Actual Power",
        line=dict(color="green")
    )
)

fig1.update_layout(
    title="Actual vs Expected Solar Generation",
    xaxis_title="Timestamp",
    yaxis_title="Power (kW)",
    template="plotly_white",
    hovermode="x unified"
)

fig1.show()

#
# CHART 2: INVERTER-WISE ENERGY PERFORMANCE
#

fig2 = go.Figure()

fig2.add_trace(
    go.Bar(
        x=inverter_summary["inverter_id"],
        y=inverter_summary["expected_energy_kwh"],
        name="Expected Energy",
        marker_color="royalblue"
    )
)

fig2.add_trace(
    go.Bar(
        x=inverter_summary["inverter_id"],
        y=inverter_summary["actual_energy_kwh"],
        name="Actual Energy",
        marker_color="seagreen"
    )
)

fig2.add_trace(
    go.Bar(
        x=inverter_summary["inverter_id"],
        y=inverter_summary["energy_shortfall_kwh"],
        name="Energy Shortfall",
        marker_color="tomato"
    )
)

fig2.update_layout(
    title="Inverter-wise Energy and Shortfall",
    xaxis_title="Inverter",
    yaxis_title="Energy (kWh)",
    barmode="group",
    template="plotly_white"
)

fig2.show()

#
# CHART 3: WEATHER IRRADIANCE VS ACTUAL POWER
#

plant_weather = performance_df.groupby(
    "timestamp"
).agg(
    actual_power_kw=("ac_power_kw", "sum"),
    irradiance_wm2=("irradiance_wm2", "mean")
).reset_index()

fig3 = make_subplots(specs=[[{"secondary_y": True}]])

fig3.add_trace(
    go.Scatter(
        x=plant_weather["timestamp"],
        y=plant_weather["irradiance_wm2"],
        name="Irradiance",
        line=dict(color="orange")
    ),
    secondary_y=False
)

fig3.add_trace(
    go.Scatter(
        x=plant_weather["timestamp"],
        y=plant_weather["actual_power_kw"],
        name="Actual Plant Power",
        line=dict(color="green")
    ),
    secondary_y=True
)

fig3.update_layout(
    title="Solar Irradiance vs Actual Plant Power",
    template="plotly_white",
    hovermode="x unified"
)

fig3.update_xaxes(title_text="Timestamp")
fig3.update_yaxes(
    title_text="Irradiance (W/m²)",
    secondary_y=False
)
fig3.update_yaxes(
    title_text="Actual Power (kW)",
    secondary_y=True
)

fig3.show()

#
# CHART 4: UNDERPERFORMANCE INCIDENT TIMELINE
#

if len(investigation_df) > 0:

    fig4 = px.scatter(
        investigation_df,
        x="timestamp",
        y="inverter_id",
        color="deficit_percentage",
        size="energy_deficit_kwh",
        hover_data=[
            "ac_power_kw",
            "expected_power_kw",
            "event_correlation"
        ],
        color_continuous_scale="Reds",
        title="Underperformance Incident Timeline"
    )

    fig4.update_layout(
        xaxis_title="Timestamp",
        yaxis_title="Inverter",
        template="plotly_white"
    )

    fig4.show()

else:
    print("No underperformance incidents to display.")

#
# SAVE CHARTS AS HTML FILES
#

import os

os.makedirs("SolarGuard/charts", exist_ok=True)

fig1.write_html(
    "SolarGuard/charts/actual_vs_expected.html"
)

fig2.write_html(
    "SolarGuard/charts/inverter_performance.html"
)

fig3.write_html(
    "SolarGuard/charts/weather_vs_power.html"
)

if len(investigation_df) > 0:
    fig4.write_html(
        "SolarGuard/charts/underperformance_timeline.html"
    )

print("\nAll interactive charts saved successfully!")
print("Location: SolarGuard/charts/")
print("STEP 9 COMPLETED")


SOLARGUARD VISUALIZATION MODULE



All interactive charts saved successfully!
Location: SolarGuard/charts/
STEP 9 COMPLETED


In [ ]:


# CREATE SQL DATABASE AND TABLES
#

import sqlite3
import pandas as pd
import os

print("SOLARGUARD SQL DATABASE SETUP")
print("=" * 55)

# Create database folder
os.makedirs("SolarGuard/database", exist_ok=True)

# Connect to SQLite database
conn = sqlite3.connect(
    "SolarGuard/database/solarguard.db"
)

print("SQL database connected successfully!")

#
# SAVE PROCESSED DATAFRAMES AS SQL TABLES
#

performance_df.to_sql(
    "performance_data",
    conn,
    if_exists="replace",
    index=False
)

investigation_df.to_sql(
    "investigation_events",
    conn,
    if_exists="replace",
    index=False
)

inverter_summary.to_sql(
    "inverter_summary",
    conn,
    if_exists="replace",
    index=False
)

event_summary.to_sql(
    "event_summary",
    conn,
    if_exists="replace",
    index=False
)

clean_alarms.to_sql(
    "equipment_alarms",
    conn,
    if_exists="replace",
    index=False
)

clean_cleaning.to_sql(
    "cleaning_records",
    conn,
    if_exists="replace",
    index=False
)

clean_maintenance.to_sql(
    "maintenance_tickets",
    conn,
    if_exists="replace",
    index=False
)

print("\nALL SQL TABLES CREATED SUCCESSFULLY!")

# Display table names
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

display(tables)

print("STEP 10.1 COMPLETED")


SOLARGUARD SQL DATABASE SETUP
SQL database connected successfully!

ALL SQL TABLES CREATED SUCCESSFULLY!


,name
0,cleaning_records
1,equipment_alarms
2,event_summary
3,inverter_summary
4,investigation_events
5,maintenance_tickets
6,performance_data


STEP 10.1 COMPLETED


In [ ]:


# SQL ANALYTICAL QUERIES
#

print("SOLARGUARD SQL ANALYTICS")
print("=" * 55)

# QUERY 1: Total inverter records
print("\n1. TOTAL INVERTER RECORDS")

query1 = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT inverter_id) AS total_inverters
FROM performance_data;
"""

display(pd.read_sql_query(query1, conn))


# QUERY 2: Inverter-wise energy production
print("\n2. INVERTER-WISE ENERGY PRODUCTION")

query2 = """
SELECT
    inverter_id,
    ROUND(SUM(actual_energy_kwh), 2) AS actual_energy,
    ROUND(SUM(expected_energy_kwh), 2) AS expected_energy
FROM performance_data
GROUP BY inverter_id
ORDER BY actual_energy DESC;
"""

display(pd.read_sql_query(query2, conn))


# QUERY 3: Underperformance observations by inverter
print("\n3. UNDERPERFORMANCE BY INVERTER")

query3 = """
SELECT
    inverter_id,
    COUNT(*) AS flagged_observations,
    ROUND(AVG(deficit_percentage), 2) AS average_deficit
FROM performance_data
WHERE performance_status = 'Underperformance'
GROUP BY inverter_id
ORDER BY flagged_observations DESC;
"""

display(pd.read_sql_query(query3, conn))


# QUERY 4: Equipment alarm summary
print("\n4. EQUIPMENT ALARM SUMMARY")

query4 = """
SELECT
    inverter_id,
    alarm_code,
    severity,
    COUNT(*) AS alarm_count
FROM equipment_alarms
GROUP BY inverter_id, alarm_code, severity
ORDER BY alarm_count DESC;
"""

display(pd.read_sql_query(query4, conn))


# QUERY 5: Maintenance ticket summary
print("\n5. MAINTENANCE TICKET SUMMARY")

query5 = """
SELECT
    inverter_id,
    maintenance_status,
    COUNT(*) AS ticket_count
FROM maintenance_tickets
GROUP BY inverter_id, maintenance_status;
"""

display(pd.read_sql_query(query5, conn))


# QUERY 6: Data quality and missing readings
print("\n6. DATA QUALITY SUMMARY")

query6 = """
SELECT
    COUNT(*) AS total_records,
    SUM(CASE
        WHEN missing_inverter_reading = 1 THEN 1
        ELSE 0
    END) AS missing_readings,
    SUM(CASE
        WHEN invalid_reading_flag = 1 THEN 1
        ELSE 0
    END) AS invalid_readings
FROM performance_data;
"""

display(pd.read_sql_query(query6, conn))


# Save and close database connection
conn.commit()

print("\nALL SQL QUERIES EXECUTED SUCCESSFULLY!")
print("STEP 10.2 COMPLETED")


SOLARGUARD SQL ANALYTICS

1. TOTAL INVERTER RECORDS


,total_records,total_inverters
0,2016,3



2. INVERTER-WISE ENERGY PRODUCTION


,inverter_id,actual_energy,expected_energy
0,INV-03,2025.34,2020.4
1,INV-01,1995.35,2020.4
2,INV-02,1886.74,2020.4



3. UNDERPERFORMANCE BY INVERTER


,inverter_id,flagged_observations,average_deficit
0,INV-02,57,36.71
1,INV-03,16,36.97
2,INV-01,15,38.53


... (3 outputs left out)

,inverter_id,maintenance_status,ticket_count
0,INV-01,Closed,1
1,INV-02,Closed,1
2,INV-02,Open,1
3,INV-03,Closed,1



6. DATA QUALITY SUMMARY


,total_records,missing_readings,invalid_readings
0,2016,8,1



ALL SQL QUERIES EXECUTED SUCCESSFULLY!
STEP 10.2 COMPLETED


In [ ]:

#
#
# DATABASE VERIFICATION
#

print("SOLARGUARD DATABASE VERIFICATION")

cursor = conn.cursor()

cursor.execute(
    "SELECT COUNT(*) FROM performance_data"
)

print(
    "Performance records stored:",
    cursor.fetchone()[0]
)

cursor.execute(
    "SELECT COUNT(*) FROM investigation_events"
)

print(
    "Investigation records stored:",
    cursor.fetchone()[0]
)

print("\nDATABASE LOCATION:")
print("SolarGuard/database/solarguard.db")




SOLARGUARD DATABASE VERIFICATION
Performance records stored: 2016
Investigation records stored: 88

DATABASE LOCATION:
SolarGuard/database/solarguard.db


In [ ]:

#
#
# INSTALL AND INITIALIZE PYSPARK
#

!pip -q install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, to_date, hour,
    when, round as spark_round, avg, sum as spark_sum,
    count, isnan
)
from pyspark.sql.types import DoubleType

# Start Spark session
spark = SparkSession.builder \
    .appName("SolarGuardAnalytics") \
    .master("local[*]") \
    .getOrCreate()

print("PySpark installed successfully!")
print("Spark version:", spark.version)
print("SolarGuard Spark session started!")


PySpark installed successfully!
Spark version: 4.0.5
SolarGuard Spark session started!


In [ ]:

#
# SOLARGUARD - STEP 11.2
# LOAD DATA INTO PYSPARK
#

# Save existing processed data temporarily
performance_df.to_csv(
    "SolarGuard/data/spark_input.csv",
    index=False
)

# Read CSV using PySpark
spark_df = spark.read.csv(
    "SolarGuard/data/spark_input.csv",
    header=True,
    inferSchema=True
)

print("DATA LOADED INTO PYSPARK SUCCESSFULLY!")

# Display schema
print("\nDATA SCHEMA:")
spark_df.printSchema()

# Display first five records
print("\nFIRST FIVE RECORDS:")
spark_df.show(5, truncate=False)

print("Total records:", spark_df.count())


DATA LOADED INTO PYSPARK SUCCESSFULLY!

DATA SCHEMA:
root
 |-- inverter_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- ac_power_kw: double (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- invalid_reading_flag: boolean (nullable = true)
 |-- missing_inverter_reading: boolean (nullable = true)
 |-- exact_weather_match: boolean (nullable = true)
 |-- irradiance_wm2: double (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- module_temperature_c: double (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- missing_weather_observation: boolean (nullable = true)
 |-- expected_power_kw: double (nullable = true)
 |-- expected_energy_kwh: double (nullable = true)
 |-- actual_energy_kwh: double (nullable = true)
 |-- power_deficit_kw: double (nullable = true)
 |-- energy_deficit_kwh: double (nullable = true)
 |-- deficit_percentage: double (nullable = true)
 |-- performance_status: string (nullable = true)



In [ ]:


# SOLARGUARD - STEP 11.3
# FEATURE PROCESSING USING PYSPARK


# Convert timestamp into Spark timestamp format
spark_df = spark_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

# Extract date and hour
spark_df = spark_df.withColumn(
    "reading_date",
    to_date(col("timestamp"))
)

spark_df = spark_df.withColumn(
    "reading_hour",
    hour(col("timestamp"))
)

# Ensure numerical columns have correct types
spark_df = spark_df.withColumn(
    "ac_power_kw",
    col("ac_power_kw").cast(DoubleType())
)

spark_df = spark_df.withColumn(
    "expected_power_kw",
    col("expected_power_kw").cast(DoubleType())
)

spark_df = spark_df.withColumn(
    "irradiance_wm2",
    col("irradiance_wm2").cast(DoubleType())
)

# Create power difference feature
spark_df = spark_df.withColumn(
    "power_difference_kw",
    col("expected_power_kw") - col("ac_power_kw")
)

# Create simplified analytical category
spark_df = spark_df.withColumn(
    "analysis_category",
    when(
        col("missing_inverter_reading") == True,
        "Missing Data"
    ).when(
        col("invalid_reading_flag") == True,
        "Invalid Data"
    ).when(
        col("irradiance_wm2") < 100,
        "Low Irradiance"
    ).when(
        col("performance_status") == "Underperformance",
        "Underperformance"
    ).otherwise(
        "Normal Operation"
    )
)

print("FEATURE PROCESSING COMPLETED!")

spark_df.select(
    "timestamp",
    "inverter_id",
    "reading_date",
    "reading_hour",
    "ac_power_kw",
    "expected_power_kw",
    "power_difference_kw",
    "analysis_category"
).show(10, truncate=False)



FEATURE PROCESSING COMPLETED!
+-------------------+-----------+------------+------------+-----------+-----------------+-------------------+-----------------+
|timestamp          |inverter_id|reading_date|reading_hour|ac_power_kw|expected_power_kw|power_difference_kw|analysis_category|
+-------------------+-----------+------------+------------+-----------+-----------------+-------------------+-----------------+
|2026-05-01 00:00:00|INV-01     |2026-05-01  |0           |0.0        |0.0              |0.0                |Low Irradiance   |
|2026-05-01 00:00:00|INV-02     |2026-05-01  |0           |0.0        |0.0              |0.0                |Low Irradiance   |
|2026-05-01 00:00:00|INV-03     |2026-05-01  |0           |0.0        |0.0              |0.0                |Low Irradiance   |
|2026-05-01 00:15:00|INV-01     |2026-05-01  |0           |0.0        |0.0              |0.0                |Low Irradiance   |
|2026-05-01 00:15:00|INV-02     |2026-05-01  |0           |0.0        |0.0

In [ ]:

#
# SOLARGUARD - STEP 11.4
# SPARK SQL ANALYTICAL QUERIES
#

# Register temporary SQL table
spark_df.createOrReplaceTempView("solar_data")

# QUERY 1: Inverter-wise power performance
print("1. INVERTER-WISE POWER PERFORMANCE")

spark_result1 = spark.sql("""
SELECT
    inverter_id,
    ROUND(AVG(ac_power_kw), 2) AS average_actual_power,
    ROUND(AVG(expected_power_kw), 2) AS average_expected_power,
    COUNT(*) AS total_readings
FROM solar_data
GROUP BY inverter_id
ORDER BY inverter_id
""")

spark_result1.show()

# QUERY 2: Count performance categories
print("2. PERFORMANCE CATEGORY SUMMARY")

spark_result2 = spark.sql("""
SELECT
    analysis_category,
    COUNT(*) AS observation_count
FROM solar_data
GROUP BY analysis_category
ORDER BY observation_count DESC
""")

spark_result2.show()

# QUERY 3: Underperformance by inverter
print("3. UNDERPERFORMANCE BY INVERTER")

spark_result3 = spark.sql("""
SELECT
    inverter_id,
    COUNT(*) AS flagged_readings,
    ROUND(AVG(deficit_percentage), 2) AS average_deficit_percentage
FROM solar_data
WHERE performance_status = 'Underperformance'
GROUP BY inverter_id
ORDER BY flagged_readings DESC
""")

spark_result3.show()

print("SPARK SQL ANALYSIS COMPLETED!")


1. INVERTER-WISE POWER PERFORMANCE
+-----------+--------------------+----------------------+--------------+
|inverter_id|average_actual_power|average_expected_power|total_readings|
+-----------+--------------------+----------------------+--------------+
|     INV-01|               11.97|                  12.1|           672|
|     INV-02|               11.26|                  12.1|           672|
|     INV-03|               12.09|                  12.1|           672|
+-----------+--------------------+----------------------+--------------+

2. PERFORMANCE CATEGORY SUMMARY
+-----------------+-----------------+
|analysis_category|observation_count|
+-----------------+-----------------+
|   Low Irradiance|             1042|
| Normal Operation|              877|
| Underperformance|               88|
|     Missing Data|                8|
|     Invalid Data|                1|
+-----------------+-----------------+

3. UNDERPERFORMANCE BY INVERTER
+-----------+----------------+----------------

In [ ]:

#
# SOLARGUARD - STEP 11.5
# SAVE PYSPARK OUTPUT
#

import os

# Save processed dataset in Parquet format
spark_df.write.mode("overwrite").parquet(
    "SolarGuard/data/spark_processed"
)

# Save summary as CSV
spark_result1.toPandas().to_csv(
    "SolarGuard/data/spark_inverter_summary.csv",
    index=False
)

spark_result2.toPandas().to_csv(
    "SolarGuard/data/spark_category_summary.csv",
    index=False
)

spark_result3.toPandas().to_csv(
    "SolarGuard/data/spark_underperformance_summary.csv",
    index=False
)

print("PYSPARK OUTPUT SAVED SUCCESSFULLY!")
print("Processed data: SolarGuard/data/spark_processed")
print("Summary files saved in SolarGuard/data/")
print("STEP 11 COMPLETED!")


PYSPARK OUTPUT SAVED SUCCESSFULLY!
Processed data: SolarGuard/data/spark_processed
Summary files saved in SolarGuard/data/
STEP 11 COMPLETED!


In [ ]:

#
# SOLARGUARD - STEP 12.1
# INSTALL STREAMLIT
#

!pip -q install streamlit

import streamlit as st
import os

print("Streamlit installed successfully!")
print("Version:", st.__version__)


Streamlit installed successfully!
Version: 1.67.0


In [ ]:

%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
from pathlib import Path

#
# SOLARGUARD - SOLAR PLANT ANALYTICS DASHBOARD
#

BASE = Path(__file__).parent
DATA = BASE / "SolarGuard" / "data"
DB = BASE / "SolarGuard" / "database" / "solarguard.db"

st.set_page_config(
    page_title="SolarGuard | Solar Analytics",
    page_icon="☀️",
    layout="wide"
)

# ---------------- CSS DESIGN ----------------

st.markdown("""
<style>
.stApp {
    background-color: #f5f7fb;
}
[data-testid="stSidebar"] {
    background-color: #eaf0f7;
}
.main-title {
    font-size: 32px;
    font-weight: 800;
    color: #17365d;
}
.subtitle {
    color: #64748b;
    font-size: 14px;
}
.kpi {
    background: white;
    border: 1px solid #dce5ef;
    border-radius: 12px;
    padding: 18px;
    min-height: 120px;
}
.kpi-label {
    color: #64748b;
    font-size: 13px;
}
.kpi-value {
    color: #17365d;
    font-size: 27px;
    font-weight: 800;
    margin-top: 8px;
}
.section-title {
    color: #17365d;
    font-size: 22px;
    font-weight: 750;
}
</style>
""", unsafe_allow_html=True)

# ---------------- DATA LOADING ----------------

@st.cache_data
def load_csv(filename):
    path = DATA / filename
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

@st.cache_data
def load_database_table(table_name):
    if not DB.exists():
        return pd.DataFrame()

    try:
        with sqlite3.connect(DB) as connection:
            return pd.read_sql_query(
                f'SELECT * FROM "{table_name}"',
                connection
            )
    except Exception:
        return pd.DataFrame()

performance = load_csv("solar_performance_analysis.csv")
investigation = load_csv("investigation_events.csv")

alarms = load_database_table("equipment_alarms")
maintenance = load_database_table("maintenance_tickets")
cleaning = load_database_table("cleaning_records")

if performance.empty:
    st.error(
        "Performance dataset not found. "
        "Please run the earlier SolarGuard Colab steps first."
    )
    st.stop()

performance["timestamp"] = pd.to_datetime(
    performance["timestamp"], errors="coerce"
)

if "inverter_id" not in performance.columns:
    st.error("The inverter_id column is missing.")
    st.stop()

# ---------------- SIDEBAR ----------------

st.sidebar.title("☀️ SolarGuard")
st.sidebar.caption("Solar Plant Analytics")

page = st.sidebar.radio(
    "NAVIGATION",
    [
        "Executive Overview",
        "Inverter Investigation",
        "Weather & Irradiance",
        "Equipment & Maintenance",
        "Data Quality Audit",
        "Investigation Report"
    ]
)

st.sidebar.divider()
st.sidebar.subheader("Analysis Filters")

inverters = sorted(
    performance["inverter_id"].dropna().unique().tolist()
)

selected_inverters = st.sidebar.multiselect(
    "Select Inverters",
    inverters,
    default=inverters
)

date_min = performance["timestamp"].min().date()
date_max = performance["timestamp"].max().date()

date_range = st.sidebar.date_input(
    "Monitoring Date Range",
    value=(date_min, date_max),
    min_value=date_min,
    max_value=date_max
)

if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
else:
    start_date, end_date = date_min, date_max

df = performance[
    performance["inverter_id"].isin(selected_inverters)
    & (performance["timestamp"].dt.date >= start_date)
    & (performance["timestamp"].dt.date <= end_date)
].copy()

if df.empty:
    st.warning("No records found for the selected filters.")
    st.stop()

# ---------------- COMMON MEASURES ----------------

actual = pd.to_numeric(
    df["actual_energy_kwh"], errors="coerce"
).sum()

expected = pd.to_numeric(
    df["expected_energy_kwh"], errors="coerce"
).sum()

shortfall = max(0, expected - actual)

if "performance_status" in df.columns:
    flagged = int(
        (df["performance_status"] == "Underperformance").sum()
    )
else:
    flagged = 0

missing = int(
    df["missing_inverter_reading"].fillna(False).astype(bool).sum()
) if "missing_inverter_reading" in df.columns else 0

# ---------------- HEADER ----------------

st.markdown(
    '<div class="main-title">☀️ SolarGuard Analytics</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Solar Plant Underperformance Investigation System'
    '</div>',
    unsafe_allow_html=True
)

st.divider()

#
# PAGE 1 - EXECUTIVE OVERVIEW
#

if page == "Executive Overview":

    st.markdown(
        '<div class="section-title">Executive Overview</div>',
        unsafe_allow_html=True
    )

    st.caption(
        "Baseline comparison and plant performance monitoring"
    )

    c1, c2, c3, c4 = st.columns(4)

    cards = [
        ("Actual Energy", f"{actual:,.2f} kWh"),
        ("Expected Energy", f"{expected:,.2f} kWh"),
        ("Energy Shortfall", f"{shortfall:,.2f} kWh"),
        ("Flagged Readings", str(flagged))
    ]

    for col, (label, value) in zip(
        [c1, c2, c3, c4], cards
    ):
        with col:
            st.markdown(
                f"""
                <div class="kpi">
                    <div class="kpi-label">{label}</div>
                    <div class="kpi-value">{value}</div>
                </div>
                """,
                unsafe_allow_html=True
            )

    st.write("")

    timeline = df.groupby("timestamp", as_index=False).agg(
        actual_power=("ac_power_kw", "sum"),
        expected_power=("expected_power_kw", "sum")
    )

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=timeline["timestamp"],
        y=timeline["expected_power"],
        name="Expected Power",
        line=dict(dash="dash")
    ))

    fig.add_trace(go.Scatter(
        x=timeline["timestamp"],
        y=timeline["actual_power"],
        name="Actual Power"
    ))

    fig.update_layout(
        title="Actual vs Expected Generation",
        xaxis_title="Timestamp",
        yaxis_title="Power (kW)",
        template="plotly_white",
        hovermode="x unified"
    )

    st.plotly_chart(fig, use_container_width=True)

    st.subheader("Inverter Performance Comparison")

    summary = df.groupby("inverter_id", as_index=False).agg(
        actual_energy=("actual_energy_kwh", "sum"),
        expected_energy=("expected_energy_kwh", "sum")
    )

    chart = px.bar(
        summary,
        x="inverter_id",
        y=["actual_energy", "expected_energy"],
        barmode="group",
        title="Actual and Expected Energy by Inverter",
        labels={
            "value": "Energy (kWh)",
            "inverter_id": "Inverter",
            "variable": "Measure"
        }
    )

    st.plotly_chart(chart, use_container_width=True)

#
# PAGE 2 - INVERTER INVESTIGATION
#

elif page == "Inverter Investigation":

    st.markdown(
        '<div class="section-title">Inverter Investigation</div>',
        unsafe_allow_html=True
    )

    if "performance_status" in df.columns:
        flagged_df = df[
            df["performance_status"] == "Underperformance"
        ].copy()
    else:
        flagged_df = pd.DataFrame()

    st.metric("Underperformance observations", len(flagged_df))

    if not flagged_df.empty:

        fig = px.scatter(
            flagged_df,
            x="timestamp",
            y="inverter_id",
            color="deficit_percentage",
            hover_data=[
                "ac_power_kw",
                "expected_power_kw"
            ],
            title="Underperformance Incident Timeline",
            color_continuous_scale="Reds"
        )

        st.plotly_chart(fig, use_container_width=True)

        st.subheader("Supporting Records")
        st.dataframe(flagged_df, use_container_width=True)

    else:
        st.info("No underperformance records for the selected filters.")

#
# PAGE 3 - WEATHER AND IRRADIANCE
#

elif page == "Weather & Irradiance":

    st.markdown(
        '<div class="section-title">Weather & Irradiance</div>',
        unsafe_allow_html=True
    )

    if "irradiance_wm2" in df.columns:

        weather = df.groupby(
            "timestamp", as_index=False
        ).agg(
            irradiance=("irradiance_wm2", "mean"),
            actual_power=("ac_power_kw", "sum")
        )

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=weather["timestamp"],
            y=weather["irradiance"],
            name="Irradiance (W/m²)"
        ))

        fig.add_trace(go.Scatter(
            x=weather["timestamp"],
            y=weather["actual_power"],
            name="Actual Power (kW)",
            yaxis="y2"
        ))

        fig.update_layout(
            title="Irradiance and Actual Power",
            template="plotly_white",
            yaxis=dict(title="Irradiance (W/m²)"),
            yaxis2=dict(
                title="Power (kW)",
                overlaying="y",
                side="right"
            )
        )

        st.plotly_chart(fig, use_container_width=True)

        st.subheader("Weather Supporting Records")
        st.dataframe(
            df[[
                "timestamp",
                "inverter_id",
                "irradiance_wm2",
                "ac_power_kw"
            ]],
            use_container_width=True
        )

    else:
        st.warning("Irradiance data is unavailable.")

#
# PAGE 4 - EQUIPMENT AND MAINTENANCE
#

elif page == "Equipment & Maintenance":

    st.markdown(
        '<div class="section-title">Equipment & Maintenance</div>',
        unsafe_allow_html=True
    )

    st.subheader("Equipment Alarms")

    if not alarms.empty:
        st.dataframe(alarms, use_container_width=True)
    else:
        st.info("No equipment alarm records found.")

    st.subheader("Maintenance Tickets")

    if not maintenance.empty:
        st.dataframe(maintenance, use_container_width=True)
    else:
        st.info("No maintenance records found.")

    st.subheader("Cleaning Records")

    if not cleaning.empty:
        st.dataframe(cleaning, use_container_width=True)
    else:
        st.info("No cleaning records found.")

    st.caption(
        "Recorded events are supporting evidence of temporal "
        "association, not proof of equipment failure."
    )

#
# PAGE 5 - DATA QUALITY AUDIT

elif page == "Data Quality Audit":

    st.markdown(
        '<div class="section-title">Data Quality Audit</div>',
        unsafe_allow_html=True
    )

    total = len(df)

    invalid = int(
        df["invalid_reading_flag"].fillna(False).astype(bool).sum()
    ) if "invalid_reading_flag" in df.columns else 0

    missing_weather = int(
        df["irradiance_wm2"].isna().sum()
    ) if "irradiance_wm2" in df.columns else 0

    st.write("### Data Quality Summary")

    q1, q2, q3, q4 = st.columns(4)

    q1.metric("Total Records", total)
    q2.metric("Missing Readings", missing)
    q3.metric("Invalid Readings", invalid)
    q4.metric("Missing Weather", missing_weather)

    st.subheader("Data Quality Records")

    quality_cols = [
        c for c in [
            "timestamp",
            "inverter_id",
            "ac_power_kw",
            "irradiance_wm2",
            "missing_inverter_reading",
            "invalid_reading_flag"
        ] if c in df.columns
    ]

    st.dataframe(
        df[quality_cols],
        use_container_width=True
    )

#
# PAGE 6 - INVESTIGATION REPORT
#

elif page == "Investigation Report":

    st.markdown(
        '<div class="section-title">Investigation Report</div>',
        unsafe_allow_html=True
    )

    st.subheader("Monitoring Summary")

    st.write("Monitoring period:", str(start_date), "to", str(end_date))
    st.write("Selected inverters:", ", ".join(selected_inverters))
    st.write("Actual energy:", round(actual, 2), "kWh")
    st.write("Expected energy:", round(expected, 2), "kWh")
    st.write("Energy shortfall:", round(shortfall, 2), "kWh")
    st.write("Flagged observations:", flagged)
    st.write("Missing readings:", missing)

    if not investigation.empty:
        filtered_investigation = investigation[
            investigation["inverter_id"].isin(selected_inverters)
        ].copy()

        st.subheader("Investigation Supporting Records")
        st.dataframe(
            filtered_investigation,
            use_container_width=True
        )
    else:
        filtered_investigation = pd.DataFrame()

    report = f"""
SOLARGUARD - SOLAR PLANT INVESTIGATION REPORT

Monitoring period: {start_date} to {end_date}

Selected inverters: {", ".join(selected_inverters)}

Actual energy: {actual:.2f} kWh
Expected energy: {expected:.2f} kWh
Positive energy shortfall: {shortfall:.2f} kWh

Underperformance observations: {flagged}
Missing readings: {missing}

INTERPRETATION:
The application compares measured inverter output against
a simplified irradiance-based theoretical baseline.

Flagged periods require further technical investigation.
Alarms and maintenance events represent temporal associations
and do not independently establish root causes.

Baseline assumptions:
Rated inverter capacity: 50 kW
Performance Ratio: 0.80

This report is based on the supplied demonstration dataset.
"""

    st.download_button(
        "Download Investigation Report",
        data=report,
        file_name="SolarGuard_Investigation_Report.txt",
        mime="text/plain"
    )

    st.download_button(
        "Download Filtered Performance Data",
        data=df.to_csv(index=False),
        file_name="SolarGuard_Performance_Data.csv",
        mime="text/csv"
    )

st.divider()

st.caption(
    "SolarGuard | Solar Plant Underperformance Investigation | "
    "Academic Demonstration Project"
)


Writing app.py


In [ ]:



# START STREAMLIT DASHBOARD IN GOOGLE COLAB


!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/solarguard.log 2>&1 &


In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
import subprocess
import time
import requests

log = open("/content/solarguard.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

try:
    response = requests.get("http://127.0.0.1:8501", timeout=10)
    print("SUCCESS! Status:", response.status_code)
except Exception as e:
    print("SERVER NOT READY:", e)
    print("\nCheck the log:")
    print(open("/content/solarguard.log").read()[-4000:])

SUCCESS! Status: 200


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [ ]:

import pandas as pd
import os

file_path = "/content/SolarGuard/data/investigation_events.csv"

if os.path.exists(file_path):
    investigation_df = pd.read_csv(file_path)

    print("SOLARGUARD INVESTIGATION DATA")
    print("=" * 50)

    print("\nDataset shape:", investigation_df.shape)

    print("\nColumn names:")
    print(investigation_df.columns.tolist())

    print("\nFirst 10 records:")
    display(investigation_df.head(10))

    print("\nDataset information:")
    investigation_df.info()
else:
    print("Investigation dataset not found!")


SOLARGUARD INVESTIGATION DATA

Dataset shape: (88, 25)

Column names:
['inverter_id', 'timestamp', 'ac_power_kw', 'energy_kwh', 'invalid_reading_flag', 'missing_inverter_reading', 'exact_weather_match', 'irradiance_wm2', 'ambient_temperature_c', 'module_temperature_c', 'weather_condition', 'missing_weather_observation', 'expected_power_kw', 'expected_energy_kwh', 'actual_energy_kwh', 'power_deficit_kw', 'energy_deficit_kwh', 'deficit_percentage', 'performance_status', 'alarm_within_60min', 'nearby_alarm_count', 'maintenance_overlap', 'related_ticket_ids', 'cleaning_within_24h', 'event_correlation']

First 10 records:


,inverter_id,timestamp,ac_power_kw,energy_kwh,invalid_reading_flag,missing_inverter_reading,exact_weather_match,irradiance_wm2,ambient_temperature_c,module_temperature_c,...,power_deficit_kw,energy_deficit_kwh,deficit_percentage,performance_status,alarm_within_60min,nearby_alarm_count,maintenance_overlap,related_ticket_ids,cleaning_within_24h,event_correlation
0,INV-01,2026-05-01 17:15:00,7.781,1.945,False,False,False,254.92,26.22,30.10,...,2.4158,0.60395,23.691746,Underperformance,False,0,False,NaN,True,No Recorded Event Nearby
1,INV-02,2026-05-01 17:15:00,7.066,1.766,False,False,False,254.92,26.22,30.10,...,3.1308,0.78270,30.703750,Underperformance,False,0,False,NaN,False,No Recorded Event Nearby
2,INV-03,2026-05-01 17:15:00,7.851,1.963,False,False,False,254.92,26.22,30.10,...,2.3458,0.58645,23.005257,Underperformance,False,0,False,NaN,False,No Recorded Event Nearby
3,INV-02,2026-05-01 17:30:00,4.973,1.243,False,False,True,161.28,26.31,28.27,...,1.4782,0.36955,22.913566,Underperformance,False,0,False,NaN,False,No Recorded Event Nearby
4,INV-03,2026-05-01 17:30:00,4.901,1.225,False,False,True,161.28,26.31,28.27,...,1.5502,0.38755,24.029638,Underperformance,False,0,False,NaN,False,No Recorded Event Nearby
5,INV-01,2026-05-01 17:45:00,2.910,0.728,False,False,False,161.28,26.31,28.27,...,3.5412,0.88530,54.892113,Underperformance,False,0,False,NaN,True,No Recorded Event Nearby
6,INV-02,2026-05-01 17:45:00,2.210,0.552,False,False,False,161.28,26.31,28.27,...,4.2412,1.06030,65.742808,Underperformance,False,0,False,NaN,False,No Recorded Event Nearby
7,INV-03,2026-05-01 17:45:00,2.177,0.544,False,False,False,161.28,26.31,28.27,...,4.2742,1.06855,66.254340,Underperformance,False,0,False,NaN,False,No Recorded Event Nearby
8,INV-01,2026-05-02 17:15:00,7.382,1.846,False,False,False,243.62,26.74,30.63,...,2.3628,0.59070,24.246778,Underperformance,False,0,False,NaN,True,No Recorded Event Nearby
9,INV-02,2026-05-02 17:15:00,7.342,1.835,False,False,False,243.62,26.74,30.63,...,2.4028,0.60070,24.657253,Underperformance,False,0,False,NaN,True,No Recorded Event Nearby



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   inverter_id                  88 non-null     object 
 1   timestamp                    88 non-null     object 
 2   ac_power_kw                  88 non-null     float64
 3   energy_kwh                   88 non-null     float64
 4   invalid_reading_flag         88 non-null     bool   
 5   missing_inverter_reading     88 non-null     bool   
 6   exact_weather_match          88 non-null     bool   
 7   irradiance_wm2               88 non-null     float64
 8   ambient_temperature_c        88 non-null     float64
 9   module_temperature_c         88 non-null     float64
 10  weather_condition            88 non-null     object 
 11  missing_weather_observation  88 non-null     bool   
 12  expected_power_kw            88 non-null     float64
 13  

In [ ]:
import shutil
shutil.copy("/content/app.py", "/content/SolarGuard/app.py")

'/content/SolarGuard/app.py'

In [ ]:
import shutil
from google.colab import files

# Create ZIP of the complete project folder
zip_path = shutil.make_archive(
    "/content/SolarGuard_Final_Project",
    "zip",
    "/content",
    "SolarGuard"
)

# Download ZIP to your computer
files.download(zip_path)

print("SolarGuard project ZIP created successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

SolarGuard project ZIP created successfully!
